In [0]:
# Databricks notebook source
# =============================================================================
# Notebook  : _silver_cdf_common  [FIXED]
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/_silver_cdf_common
# Purpose   : All shared logic for Bronze CDF → Silver base tables.
#
# BUG FIXES APPLIED (vs original):
#   BUG 1  — process_batch: `df.isEmpty()` called on streaming DataFrame
#             → replaced with batch_df row count check via len(batch_df.head(1))
#   BUG 2  — process_batch: filter on `_change_type` may silently drop ALL rows
#             when CDF is freshly enabled and only "insert" rows exist but
#             starting_version is set too high. Added explicit logging so you
#             can see what change types arrived.
#   BUG 3  — process_batch: physical_cols dedup — a column can appear in BOTH
#             physical_cols and map_candidates if CDF_CORE_COLS and a_ prefix
#             logic both match. Fixed with explicit set-based dedup.
#   BUG 4  — process_batch: `keep` list filters with `c in df.columns` AFTER
#             columns have been renamed/dropped, causing KeyErrors or silently
#             dropping columns. Fixed: derive `keep` before any transforms.
#   BUG 5  — process_batch: MERGE ON clause uses source_key_value alone which
#             isn't unique across customers. Multi-tenant silver requires
#             tenant in the MERGE key. Fixed: added tenant to MERGE ON.
#   BUG 6  — run_cdf_stream: when starting_version='0' and CDF was just enabled,
#             get_cdf_start_version returns the ENABLE version, but the very
#             first write to the table happened before CDF was enabled, so the
#             stream misses all prior inserts. Fixed: when no checkpoint exists,
#             fall back to version 0 with a full-snapshot read instead of CDF.
#   BUG 7  — evolve_silver_schema: uses f.dataType.simpleString() which returns
#             types like "map<string,string>" — not valid Spark SQL DDL.
#             Fixed: map to safe DDL type strings.
#   BUG 8  — Bronze table name for "events" object: bronze_table("events") likely
#             resolves to bronze.cust_001.crm_events (wrong) instead of
#             bronze.cust_001.events_raw. Fixed in silver_events notebook;
#             object_name set to "events_raw" so helper resolves correctly.
#             (See FIXED_02a_silver_events.py)
#   BUG 9  — STANDARD_COLS_BY_OBJECT lookup uses lowercase obj_name "account"
#             but a_ columns from bronze may arrive as "a_Name", "a_Website" etc.
#             The bare = cl[2:] strip works correctly only if input is lowercase.
#             Added .lower() normalisation on all bronze column names before
#             classification.
#   BUG 10 — safe_spark_conf called with SPARK_CONF_SILVER_CDF which is defined
#             in pipeline_config — if that dict is empty or None the function
#             silently does nothing. Added a guard.
# =============================================================================

import time
from pyspark.sql import functions as F

# =============================================================================
# 0. Serverless-safe Spark config
# =============================================================================
def safe_spark_conf(configs: dict):
    if not configs:
        return
    for k, v in configs.items():
        try:
            spark.conf.set(k, v)
        except Exception:
            pass

safe_spark_conf(SPARK_CONF_SILVER_CDF)   # from pipeline_config

# =============================================================================
# 1. Standard columns per object
# =============================================================================
STANDARD_COLS_BY_OBJECT = {
    ("salesforce", "account"): {
        "name", "accountnumber", "ownerid", "site", "accountsource",
        "annualrevenue", "billingaddress", "cleanstatus", "createdbyid",
        "dandbcompanyid", "dunsnumber", "jigsaw", "description",
        "numberofemployees", "fax", "industry", "lastmodifiedbyid",
        "naicscode", "naicsdesc", "operatinghoursid", "ownership",
        "parentid", "phone", "rating", "shippingaddress", "sic", "sicdesc",
        "tickersymbol", "tradestyle", "type", "website", "yearstarted"
    },
    ("salesforce", "contact"): {
        "accountid", "assistantname", "assistantphone", "birthdate",
        "buyerattributes", "cleanstatus", "ownerid", "contactsource",
        "jigsaw", "department", "description", "donotcall", "email",
        "hasoptedoutofemail", "fax", "hasoptedoutoffax", "genderidentity",
        "homephone", "individualid", "lastmodifiedbyid", "lastcurequestdate",
        "lastcuupdatedate", "leadsource", "mailingaddress", "mobilephone",
        "name", "otheraddress", "otherphone", "phone", "pronouns",
        "reportstoid", "title"
    },
    ("salesforce", "opportunity"): {
        "accountid", "amount", "closedate", "contractid", "createdbyid",
        "description", "expectedrevenue", "forecastcategoryname",
        "lastmodifiedbyid", "leadsource", "nextstep", "name", "ownerid",
        "iqscore", "pricebook2id", "campaignid", "isprivate", "probability",
        "totalopportunityquantity", "stagename", "type"
    },
    ("salesforce", "task"): {
        "whoid", "whatid", "subject", "activitydate", "status", "priority",
        "ishighpriority", "ownerid", "description", "isdeleted", "accountid",
        "isclosed", "calldurationinseconds", "calltype", "tasksubtype"
    },
    ("salesforce", "campaign"): {
        "isactive", "name", "type", "status", "startdate", "enddate",
        "expectedrevenue", "budgetedcost", "actualcost",
        "expectedresponse", "numbersent"
    },
    ("salesforce", "campaignmember"): {
        "campaignid", "leadid", "contactid", "status", "hasresponded"
    },
    ("salesforce", "user"): {
        "email", "username", "firstname", "lastname", "isactive",
        "profileid", "userroleid", "department", "title", "phone",
        "mobilephone", "alias", "communitynickname", "usertype",
        "languagelocalkey", "localesidkey", "timezonesidkey"
    },
    ("bigquery", "events"): {
        "event", "event_text", "event_timestamp", "domain"
    },
}

CDF_CORE_COLS = frozenset({
    "tenant", "id", "source_system", "source_system_object",
    "source_key_name", "source_key_value", "data_timestamp", "status",
    "domain", "name", "email",
    "contact_source_system_object", "contact_source_key_value",
    "campaign_source_key_value", "event_timestamp",
    "a_accountid", "a_subject", "a_amount", "a_stagename",
    "event", "event_text",
})

# =============================================================================
# 2. Schema Evolution
# FIX (BUG 7): simpleString() returns internal Spark type names that are not
# valid SQL DDL (e.g. "map<string,string>"). We map to safe SQL types.
# =============================================================================
def _safe_ddl_type(spark_type) -> str:
    """Convert a Spark DataType to a safe SQL DDL string."""
    from pyspark.sql import types as T
    if isinstance(spark_type, T.StringType):    return "STRING"
    if isinstance(spark_type, T.LongType):      return "BIGINT"
    if isinstance(spark_type, T.IntegerType):   return "INT"
    if isinstance(spark_type, T.DoubleType):    return "DOUBLE"
    if isinstance(spark_type, T.FloatType):     return "FLOAT"
    if isinstance(spark_type, T.BooleanType):   return "BOOLEAN"
    if isinstance(spark_type, T.TimestampType): return "TIMESTAMP"
    if isinstance(spark_type, T.DateType):      return "DATE"
    if isinstance(spark_type, T.MapType):       return "MAP<STRING,STRING>"
    # All other types fall back to STRING (safe)
    return "STRING"

def evolve_silver_schema(df_schema, target_full: str):
    """ADD COLUMNS to silver table for any new fields arriving from bronze."""
    existing = {f.name.lower() for f in spark.table(target_full).schema}
    new_cols = [
        f"`{f.name}` {_safe_ddl_type(f.dataType)}"
        for f in df_schema
        if f.name.lower() not in existing and not f.name.startswith("_")
    ]
    if new_cols:
        print(f"  Schema evolution ({target_full}): +{len(new_cols)} col(s): {new_cols}")
        spark.sql(f"ALTER TABLE {target_full} ADD COLUMNS ({', '.join(new_cols)})")

# =============================================================================
# 3. CDF Enable Helper
# =============================================================================
def enable_cdf(bronze_full: str):
    """Enable Change Data Feed on a bronze table (idempotent)."""
    try:
        spark.sql(f"""
            ALTER TABLE {bronze_full}
            SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
        """)
        print(f"  CDF enabled: {bronze_full}")
    except Exception as e:
        print(f"  CDF note ({bronze_full}): {e}")

# =============================================================================
# 3b. Auto-detect CDF-enabled version from table history
# =============================================================================
def get_cdf_start_version(bronze_full: str) -> str:
    try:
        hist_df = spark.sql(f"DESCRIBE HISTORY {bronze_full}")
        cdf_rows = (
            hist_df
            .filter(
                (F.col("operation") == "SET TBLPROPERTIES")
                & F.col("operationParameters.properties").contains("enableChangeDataFeed")
            )
            .orderBy("version")
            .select("version")
            .collect()
        )
        if cdf_rows:
            version = str(cdf_rows[0]["version"])
            print(f"  Auto-detected CDF start version for {bronze_full}: {version}")
            return version
    except Exception as e:
        print(f"  Could not auto-detect CDF version for {bronze_full}: {e}")
    return "0"

# =============================================================================
# 3c. FIX (BUG 6): Check whether a streaming checkpoint already exists.
# If no checkpoint → this is the first run → we do a full snapshot load
# via batch read (not CDF stream) to capture all history before CDF was enabled.
# =============================================================================
def checkpoint_exists(chk_path: str) -> bool:
    try:
        files = dbutils.fs.ls(chk_path)
        return len(files) > 0
    except Exception:
        return False

def full_snapshot_load(bronze_full: str, target_full: str,
                       source_sys: str, obj_name: str):
    """
    One-time full load of all existing bronze rows into silver.
    Called only on first run (no checkpoint). Uses foreachBatch logic
    but reads the entire table as a static DataFrame.
    FIX (BUG 6): CDF cannot go back before it was enabled. A full snapshot
    read captures all history, then the CDF stream picks up from there.
    """
    print(f"  First run detected — performing full snapshot load of {bronze_full}")
    df = spark.table(bronze_full)

    # Simulate the _change_type column that CDF would provide
    df = df.withColumn("_change_type", F.lit("insert"))

    processor = make_batch_processor(source_sys, obj_name, target_full)
    processor(df, 0)
    print(f"  Snapshot load complete → {target_full}")

# =============================================================================
# 4. Silver Table Creation
# =============================================================================
def create_silver_table(table_full: str, silver_tbl_name: str):
    """Creates the silver Delta table using DDL from pipeline_config."""
    schema_def = SILVER_DDL_COLUMNS[silver_tbl_name]   # from pipeline_config
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_full}
        ({schema_def})
        USING DELTA
        CLUSTER BY (source_system, source_system_object,
                    source_key_name, source_key_value)
        {DELTA_TBLPROPS_SILVER}
    """)
    print(f"  Silver table ready: {table_full}")

# =============================================================================
# 5. foreachBatch factory
#    FIX (BUG 1): Replaced df.isEmpty() (invalid on streaming DF in some
#                 Spark versions) with len(batch_df.head(1)) == 0.
#    FIX (BUG 3): Set-based dedup of physical_cols vs map_candidates.
#    FIX (BUG 4): Derive `keep` list before any column transforms.
#    FIX (BUG 5): Added `tenant` to the MERGE ON key for multi-tenant isolation.
#    FIX (BUG 9): All column names lowercased before classification.
# =============================================================================
def make_batch_processor(source_sys: str, obj_name: str, target_full: str):
    std_cols = STANDARD_COLS_BY_OBJECT.get((source_sys, obj_name), set())
    view_sfx = f"{source_sys}_{obj_name}".replace("-", "_")

    def process_batch(batch_df, batch_id):
        # FIX (BUG 1): safe isEmpty check that works in both batch and streaming
        if len(batch_df.head(1)) == 0:
            print(f"  Batch {batch_id}: empty, skipping")
            return

        # FIX (BUG 2): log change types so empty-silver issues are diagnosable
        if "_change_type" in batch_df.columns:
            change_type_counts = (
                batch_df.groupBy("_change_type").count().collect()
            )
            print(f"  Batch {batch_id} change types: {change_type_counts}")
            df = batch_df.filter(
                F.col("_change_type").isin("insert", "update_postimage")
            )
        else:
            # Full snapshot load path (no CDF column)
            df = batch_df

        if len(df.head(1)) == 0:
            print(f"  Batch {batch_id}: no insert/update_postimage rows, skipping")
            return

        # ── Column classification ──────────────────────────────────────────
        # FIX (BUG 9): normalise to lowercase before classification
        map_candidates  = []
        physical_cols   = []
        seen            = set()   # FIX (BUG 3): prevent duplicates

        for c in df.columns:
            if c.startswith("_"):
                continue                      # CDF internal columns — drop
            cl = c.lower()
            if cl in seen:
                continue
            seen.add(cl)

            if cl in CDF_CORE_COLS:
                physical_cols.append(c)
            elif cl.startswith("a_"):
                bare = cl[2:]                 # strip "a_" prefix
                if bare in std_cols:
                    physical_cols.append(c)   # standard → keep physical
                else:
                    map_candidates.append(c)  # non-standard → pack into MAP
            else:
                physical_cols.append(c)

        # FIX (BUG 4): record which physical cols actually exist BEFORE transforms
        physical_cols_present = [c for c in physical_cols if c in df.columns]

        # ── Pack non-standard a_ columns into MAP ──────────────────────────
        if map_candidates:
            kvs = []
            for c in map_candidates:
                kvs.extend([F.lit(c), F.col(c).cast("string")])
            df = df.withColumn("custom_metadata", F.create_map(*kvs))
        else:
            df = df.withColumn("custom_metadata", F.expr("map()"))

        keep     = physical_cols_present + ["custom_metadata"]
        df_final = df.select(*keep)

        # ── Driver-level schema evolution ──────────────────────────────────
        evolve_silver_schema(df_final.schema, target_full)

        # ── MERGE ──────────────────────────────────────────────────────────
        view_name = f"silver_cdc_{view_sfx}_{batch_id}_{int(time.time())}"
        df_final.createOrReplaceTempView(view_name)

        mc  = [c for c in df_final.columns if not c.startswith("_")]
        # FIX (BUG 5): include tenant in PK set for multi-tenant MERGE isolation
        pk  = {"tenant", "id", "source_system", "source_system_object",
               "source_key_name", "source_key_value"}
        upd = ", ".join(
            f"target.`{c}` = source.`{c}`" for c in mc if c not in pk
        )
        ic  = ", ".join(f"`{c}`" for c in mc)
        iv  = ", ".join(f"source.`{c}`" for c in mc)

        # Only generate UPDATE SET if there are non-PK columns to update
        if upd:
            merge_sql = f"""
                MERGE INTO {target_full} AS target
                USING {view_name} AS source
                ON  target.tenant               = source.tenant
                AND target.source_system        = source.source_system
                AND target.source_system_object = source.source_system_object
                AND target.source_key_name      = source.source_key_name
                AND target.source_key_value     = source.source_key_value
                WHEN MATCHED
                  AND source.data_timestamp >= target.data_timestamp
                THEN UPDATE SET {upd}
                WHEN NOT MATCHED THEN
                  INSERT ({ic}) VALUES ({iv})
            """
        else:
            merge_sql = f"""
                MERGE INTO {target_full} AS target
                USING {view_name} AS source
                ON  target.tenant               = source.tenant
                AND target.source_system        = source.source_system
                AND target.source_system_object = source.source_system_object
                AND target.source_key_name      = source.source_key_name
                AND target.source_key_value     = source.source_key_value
                WHEN NOT MATCHED THEN
                  INSERT ({ic}) VALUES ({iv})
            """

        spark.sql(merge_sql)
        print(f"  Batch {batch_id} merged → {target_full}")

    return process_batch

# =============================================================================
# 6. CDF Stream Launcher
# FIX (BUG 6): On first run (no checkpoint), do a full snapshot load first,
# then start the CDF stream from the current table version so it only picks
# up future changes — avoiding DELTA_MISSING_CHANGE_DATA errors entirely.
# =============================================================================
def run_cdf_stream(bronze_full: str, target_full: str,
                   source_sys: str, obj_name: str,
                   chk_path: str, starting_version: str = "0"):

    is_first_run = not checkpoint_exists(chk_path)

    if is_first_run:
        # Full snapshot load captures all history before CDF was enabled
        full_snapshot_load(bronze_full, target_full, source_sys, obj_name)
        # Start CDF stream from NOW (current version) to pick up future changes
        effective_version = spark.sql(
            f"SELECT MAX(version) FROM (DESCRIBE HISTORY {bronze_full})"
        ).collect()[0][0]
        effective_version = str(effective_version) if effective_version is not None else "0"
        print(f"  First run complete. CDF stream will start from version {effective_version}")
    else:
        # Checkpoint exists → stream resumes automatically, version ignored
        effective_version = starting_version
        if starting_version == "0":
            effective_version = get_cdf_start_version(bronze_full)
        print(f"  Checkpoint found. Resuming CDF stream from version {effective_version}")

    stream = (
        spark.readStream
             .format("delta")
             .option("readChangeFeed",     "true")
             .option("startingVersion",    effective_version)
             .option("maxFilesPerTrigger", CDF_MAX_FILES_PER_TRIGGER)
             .table(bronze_full)
    )
    query = (
        stream.writeStream
              .foreachBatch(make_batch_processor(source_sys, obj_name, target_full))
              .option("checkpointLocation", chk_path)
              .trigger(availableNow=True)
              .start()
    )
    query.awaitTermination()
    print(f"  CDF stream complete: {bronze_full} → {target_full}")

print("_silver_cdf_common loaded.") 

In [0]:
# # =============================================================================
# # Notebook  : _silver_cdf_common
# # Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/_silver_cdf_common
# # Purpose   : All shared logic for Bronze CDF → Silver base tables.
# #             Every silver CDF notebook does:
# #               Cell 1: %run ../config/pipeline_config
# #               Cell 2: %run ./_silver_cdf_common
# #
# # REQUIRES pipeline_config to be %run FIRST — uses:
# #   SPARK_CONF_SILVER_CDF, DELTA_TBLPROPS_SILVER, SILVER_DDL_COLUMNS,
# #   CDF_MAX_FILES_PER_TRIGGER, sv, silver_catalog, silver_schema
# # =============================================================================

# import time
# from pyspark.sql import functions as F

# # =============================================================================
# # 0. Serverless-safe Spark config
# # =============================================================================
# def safe_spark_conf(configs: dict):
#     for k, v in configs.items():
#         try:
#             spark.conf.set(k, v)
#         except Exception:
#             pass

# safe_spark_conf(SPARK_CONF_SILVER_CDF)   # from pipeline_config

# # =============================================================================
# # 1. Standard columns per object
# #    Fields in this set are kept as physical columns in silver.
# #    All other a_ fields are packed into custom_metadata MAP<STRING,STRING>.
# #    This prevents schema explosion when Salesforce adds new custom fields.
# # =============================================================================
# STANDARD_COLS_BY_OBJECT = {
#     ("salesforce", "account"): {
#         "name", "accountnumber", "ownerid", "site", "accountsource",
#         "annualrevenue", "billingaddress", "cleanstatus", "createdbyid",
#         "dandbcompanyid", "dunsnumber", "jigsaw", "description",
#         "numberofemployees", "fax", "industry", "lastmodifiedbyid",
#         "naicscode", "naicsdesc", "operatinghoursid", "ownership",
#         "parentid", "phone", "rating", "shippingaddress", "sic", "sicdesc",
#         "tickersymbol", "tradestyle", "type", "website", "yearstarted"
#     },
#     ("salesforce", "contact"): {
#         "accountid", "assistantname", "assistantphone", "birthdate",
#         "buyerattributes", "cleanstatus", "ownerid", "contactsource",
#         "jigsaw", "department", "description", "donotcall", "email",
#         "hasoptedoutofemail", "fax", "hasoptedoutoffax", "genderidentity",
#         "homephone", "individualid", "lastmodifiedbyid", "lastcurequestdate",
#         "lastcuupdatedate", "leadsource", "mailingaddress", "mobilephone",
#         "name", "otheraddress", "otherphone", "phone", "pronouns",
#         "reportstoid", "title"
#     },
#     ("salesforce", "opportunity"): {
#         "accountid", "amount", "closedate", "contractid", "createdbyid",
#         "description", "expectedrevenue", "forecastcategoryname",
#         "lastmodifiedbyid", "leadsource", "nextstep", "name", "ownerid",
#         "iqscore", "pricebook2id", "campaignid", "isprivate", "probability",
#         "totalopportunityquantity", "stagename", "type"
#     },
#     ("salesforce", "task"): {
#         "whoid", "whatid", "subject", "activitydate", "status", "priority",
#         "ishighpriority", "ownerid", "description", "isdeleted", "accountid",
#         "isclosed", "calldurationinseconds", "calltype", "tasksubtype"
#     },
#     ("salesforce", "campaign"): {
#         "isactive", "name", "type", "status", "startdate", "enddate",
#         "expectedrevenue", "budgetedcost", "actualcost",
#         "expectedresponse", "numbersent"
#     },
#     ("salesforce", "campaignmember"): {
#         "campaignid", "leadid", "contactid", "status", "hasresponded"
#     },
#     ("salesforce", "user"): {
#         "email", "username", "firstname", "lastname", "isactive",
#         "profileid", "userroleid", "department", "title", "phone",
#         "mobilephone", "alias", "communityNickname", "userType",
#         "languageLocaleKey", "localeSidKey", "timeZoneSidKey"
#     },
#     ("bigquery", "events"): {
#         "event", "event_text", "event_timestamp", "domain"
#     },
# }

# # Core tracking columns — always kept as physical columns, never packed into MAP
# CDF_CORE_COLS = frozenset({
#     "tenant", "id", "source_system", "source_system_object",
#     "source_key_name", "source_key_value", "data_timestamp", "status",
#     "domain", "name", "email",
#     "contact_source_system_object", "contact_source_key_value",
#     "campaign_source_key_value", "event_timestamp",
#     "a_accountid", "a_subject", "a_amount", "a_stagename",
#     "event", "event_text",
# })

# # =============================================================================
# # 2. Schema Evolution (driver-level — never inside executor)
# # =============================================================================
# def evolve_silver_schema(df_schema, target_full: str):
#     """ADD COLUMNS to silver table for any new fields arriving from bronze."""
#     existing = {f.name.lower() for f in spark.table(target_full).schema}
#     new_cols = [
#         f"`{f.name}` {f.dataType.simpleString()}"
#         for f in df_schema
#         if f.name.lower() not in existing and not f.name.startswith("_")
#     ]
#     if new_cols:
#         print(f"  Schema evolution ({target_full}): +{len(new_cols)} col(s)")
#         spark.sql(f"ALTER TABLE {target_full} ADD COLUMNS ({', '.join(new_cols)})")

# # =============================================================================
# # 3. CDF Enable Helper
# # =============================================================================
# def enable_cdf(bronze_full: str):
#     """Enable Change Data Feed on a bronze table (idempotent)."""
#     try:
#         spark.sql(f"""
#             ALTER TABLE {bronze_full}
#             SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
#         """)
#         print(f"  CDF enabled: {bronze_full}")
#     except Exception as e:
#         print(f"  CDF note ({bronze_full}): {e}")

# # =============================================================================
# # 3b. Auto-detect CDF-enabled version from table history
# # =============================================================================
# def get_cdf_start_version(bronze_full: str) -> str:
#     """
#     Query DESCRIBE HISTORY to find the version where CDF was first enabled.
#     Looks for SET TBLPROPERTIES operations that include enableChangeDataFeed.
#     Returns the version as a string, or '0' as fallback.
#     """
#     try:
#         hist_df = spark.sql(f"DESCRIBE HISTORY {bronze_full}")
#         cdf_rows = (
#             hist_df
#             .filter(
#                 (F.col("operation") == "SET TBLPROPERTIES")
#                 & F.col("operationParameters.properties").contains("enableChangeDataFeed")
#             )
#             .orderBy("version")
#             .select("version")
#             .collect()
#         )
#         if cdf_rows:
#             version = str(cdf_rows[0]["version"])
#             print(f"  Auto-detected CDF start version for {bronze_full}: {version}")
#             return version
#     except Exception as e:
#         print(f"  Could not auto-detect CDF version for {bronze_full}: {e}")
#     return "0"

# # =============================================================================
# # 4. Silver Table Creation
# # =============================================================================
# def create_silver_table(table_full: str, silver_tbl_name: str):
#     """Creates the silver Delta table using DDL from pipeline_config."""
#     schema_def = SILVER_DDL_COLUMNS[silver_tbl_name]   # from pipeline_config
#     spark.sql(f"""
#         CREATE TABLE IF NOT EXISTS {table_full}
#         ({schema_def})
#         USING DELTA
#         CLUSTER BY (source_system, source_system_object,
#                     source_key_name, source_key_value)
#         {DELTA_TBLPROPS_SILVER}
#     """)
#     print(f"  Silver table ready: {table_full}")

# # =============================================================================
# # 5. foreachBatch factory
# #    Returns a closure bound to the object's source/target context.
# #    Each silver CDF notebook creates one closure for its specific object.
# # =============================================================================
# def make_batch_processor(source_sys: str, obj_name: str, target_full: str):
#     """
#     Returns a foreachBatch handler for the given source/object/target combo.
#     Classification:
#       - CDF metadata cols (_change_type etc.) → dropped
#       - Core tracking cols → kept as physical columns
#       - a_ cols in STANDARD_COLS → kept as physical columns
#       - a_ cols NOT in STANDARD_COLS → packed into custom_metadata MAP
#       - Other raw cols → kept as physical columns
#     """
#     std_cols = STANDARD_COLS_BY_OBJECT.get((source_sys, obj_name), set())
#     view_sfx = f"{source_sys}_{obj_name}".replace("-", "_")

#     def process_batch(batch_df, batch_id):
#         if batch_df.isEmpty():
#             return

#         # Only process inserts and post-image updates (ignore pre-image + deletes at silver)
#         df = batch_df.filter(
#             F.col("_change_type").isin("insert", "update_postimage")
#         )
#         if df.isEmpty():
#             return

#         # ── Column classification ──────────────────────────────────────────
#         map_candidates = []
#         physical_cols  = []

#         for c in df.columns:
#             if c.startswith("_"):                    # CDF internal columns
#                 continue
#             cl = c.lower()
#             if cl in CDF_CORE_COLS:
#                 physical_cols.append(c)
#             elif cl.startswith("a_"):
#                 bare = cl[2:]                        # strip "a_" prefix
#                 if bare in std_cols:
#                     physical_cols.append(c)          # standard → keep physical
#                 else:
#                     map_candidates.append(c)          # non-standard → pack into MAP
#             else:
#                 physical_cols.append(c)

#         # ── Pack non-standard a_ columns into MAP ─────────────────────────
#         if map_candidates:
#             kvs = []
#             for c in map_candidates:
#                 kvs.extend([F.lit(c), F.col(c).cast("string")])
#             df = df.withColumn("custom_metadata", F.create_map(*kvs))
#         else:
#             df = df.withColumn("custom_metadata", F.expr("map()"))

#         keep     = [c for c in physical_cols if c in df.columns] + ["custom_metadata"]
#         df_final = df.select(*keep)

#         # ── Driver-level schema evolution ─────────────────────────────────
#         evolve_silver_schema(df_final.schema, target_full)

#         # ── MERGE ──────────────────────────────────────────────────────────
#         view_name = f"silver_cdc_{view_sfx}_{batch_id}_{int(time.time())}"
#         df_final.createOrReplaceTempView(view_name)

#         mc  = [c for c in df_final.columns if not c.startswith("_")]
#         pk  = {"tenant", "id", "source_system", "source_system_object",
#                "source_key_name", "source_key_value"}
#         upd = ", ".join(
#             f"target.`{c}` = source.`{c}`" for c in mc if c not in pk
#         )
#         ic  = ", ".join(f"`{c}`" for c in mc)
#         iv  = ", ".join(f"source.`{c}`" for c in mc)

#         spark.sql(f"""
#             MERGE INTO {target_full} AS target
#             USING {view_name} AS source
#             ON  target.source_system        = source.source_system
#             AND target.source_system_object = source.source_system_object
#             AND target.source_key_name      = source.source_key_name
#             AND target.source_key_value     = source.source_key_value
#             WHEN MATCHED
#               AND source.data_timestamp >= target.data_timestamp
#             THEN UPDATE SET {upd}
#             WHEN NOT MATCHED THEN
#               INSERT ({ic}) VALUES ({iv})
#         """)
#         print(f"  Batch {batch_id} merged → {target_full}")

#     return process_batch

# # =============================================================================
# # 6. CDF Stream Launcher
# # =============================================================================
# def run_cdf_stream(bronze_full: str, target_full: str,
#                    source_sys: str, obj_name: str,
#                    chk_path: str, starting_version: str = "0"):
#     """
#     Reads CDF from bronze table; merges into silver target.
#     trigger(availableNow=True) = batch mode: process all queued versions then stop.
#     After first run, checkpoint takes over — starting_version is ignored.

#     When starting_version is '0', auto-detects the version where CDF was
#     enabled via DESCRIBE HISTORY to avoid DELTA_MISSING_CHANGE_DATA errors.
#     """
#     effective_version = starting_version
#     if starting_version == "0":
#         effective_version = get_cdf_start_version(bronze_full)

#     stream = (
#         spark.readStream
#              .format("delta")
#              .option("readChangeFeed",     "true")
#              .option("startingVersion",    effective_version)
#              .option("maxFilesPerTrigger", CDF_MAX_FILES_PER_TRIGGER)
#              .table(bronze_full)
#     )
#     query = (
#         stream.writeStream
#               .foreachBatch(make_batch_processor(source_sys, obj_name, target_full))
#               .option("checkpointLocation", chk_path)
#               .trigger(availableNow=True)
#               .start()
#     )
#     query.awaitTermination()
#     print(f"  CDF stream complete: {bronze_full} → {target_full}")

# print("_silver_cdf_common loaded.")